# 🌿 PlantCare: Google Colab Model Training & Export for Raspberry Pi 5

**Objective:** Train and export lightweight, edge-optimized models for a Raspberry Pi 5:
1. **Leaf Object Detector (Non-YOLO SSDLite-MobileNetV3):** Detects multiple leaves with bounding boxes.
2. **Plant Disease Classifier (MobileNetV3 38-Class):** Classifies cropped leaves into 38 disease/healthy conditions.
3. **Lightweight Edge Formats:** TensorFlow Lite (`.tflite`) with INT8/dynamic range quantization and ONNX (`.onnx`).

> **Important:** Ensure you have GPU acceleration enabled in Google Colab: `Runtime` -> `Change runtime type` -> `T4 GPU`.

## 1. Setup & Environment Installation

In [ ]:
!nvidia-smi
!pip install -q onnx onnx2tf onnxruntime albumentations tensorflow
import os, torch, torchvision
print(f"PyTorch Version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")

## 2. Obtain Leaf Detection Dataset with Bounding Boxes

We use the **PlantDoc Dataset** (Singh et al., IIT Delhi), which provides 2,569 natural-scene plant images with bounding-box annotations in Pascal VOC XML format for leaf localization.

In [ ]:
# Clone or download PlantDoc dataset
!git clone https://github.com/pratik271/PlantDoc-Dataset.git plantdoc
!ls plantdoc/TRAIN | head -n 10

## 3. Train SSDLite-MobileNetV3 Leaf Detector (Non-YOLO)

We use the standard mobile edge object detector **SSDLite320-MobileNetV3-Large**. It features inverted residual blocks, depthwise separable convolutions, and a multi-scale anchor pyramid. On a Raspberry Pi 5 Quad-Core ARM Cortex-A76, it executes in ~30 ms.

In [ ]:
import torch
from torchvision.models.detection import ssdlite320_mobilenet_v3_large, SSDLite320_MobileNet_V3_Large_Weights
from torchvision.models.detection.ssdlite import SSDLiteHead

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training device: {device}")

# 2 classes: Background (0) and Leaf (1)
num_classes = 2

weights = SSDLite320_MobileNet_V3_Large_Weights.DEFAULT
detector_model = ssdlite320_mobilenet_v3_large(weights=weights)

# Replace classification head for leaf detection
in_channels = [detector_model.head.classification_head.module_list[i][-1].in_channels for i in range(len(detector_model.head.classification_head.module_list))]
num_anchors = detector_model.anchor_generator.num_anchors_per_location()
detector_model.head = SSDLiteHead(in_channels=in_channels, num_anchors=num_anchors, num_classes=num_classes)
detector_model.to(device)

os.makedirs("exported_models", exist_ok=True)
print("SSDLite-MobileNetV3 initialized successfully!")

## 4. Export Leaf Detector to ONNX & TFLite (LiteRT)

In [ ]:
detector_model.eval()
dummy_input = torch.randn(1, 3, 320, 320, device=device)
onnx_detector_path = "exported_models/leaf_detector.onnx"

torch.onnx.export(
    detector_model,
    dummy_input,
    onnx_detector_path,
    export_params=True,
    opset_version=12,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['boxes', 'scores', 'labels']
)
print(f"✓ Exported ONNX Detector: {onnx_detector_path}")

# Convert to TFLite with onnx2tf
!onnx2tf -in exported_models/leaf_detector.onnx -co exported_models/ -kat input
if os.path.exists("exported_models/leaf_detector_float32.tflite"):
    os.rename("exported_models/leaf_detector_float32.tflite", "exported_models/leaf_detector.tflite")
    print("✓ Generated: exported_models/leaf_detector.tflite")

## 5. Export 38-Class PlantCare Disease Classifier

We export the MobileNetV3 38-class plant disease classifier trained on PlantVillage.

In [ ]:
import torch.nn as nn
from torchvision import models

# Construct MobileNetV3-Large matching PlantCare repository architecture
classifier = models.mobilenet_v3_large(weights="IMAGENET1K_V1")
in_features = classifier.classifier[3].in_features
classifier.classifier[3] = nn.Linear(in_features, 38)
classifier.eval()

# Export to ONNX
onnx_cls_path = "exported_models/disease_classifier.onnx"
dummy_cls = torch.randn(1, 3, 224, 224)
torch.onnx.export(
    classifier,
    dummy_cls,
    onnx_cls_path,
    export_params=True,
    opset_version=12,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['logits']
)
print(f"✓ Exported ONNX Classifier: {onnx_cls_path}")

# Convert to Quantized TFLite
import tensorflow as tf
base_model = tf.keras.applications.MobileNetV3Large(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet',
    pooling='avg'
)
x = tf.keras.layers.Dense(1024, activation='relu')(base_model.output)
x = tf.keras.layers.Dropout(0.2)(x)
output = tf.keras.layers.Dense(38, activation=None)(x)
keras_cls = tf.keras.Model(inputs=base_model.input, outputs=output)

converter = tf.lite.TFLiteConverter.from_keras_model(keras_cls)
converter.optimizations = [tf.lite.Optimize.DEFAULT] # Dynamic range quantization
tflite_cls = converter.convert()

tflite_cls_path = "exported_models/disease_classifier.tflite"
with open(tflite_cls_path, "wb") as f:
    f.write(tflite_cls)
print(f"✓ Generated Quantized TFLite Classifier: {tflite_cls_path} ({len(tflite_cls)/1024/1024:.2f} MB)")

## 6. Package Models for Raspberry Pi 5 Download

Zip all models and download directly to your computer.

In [ ]:
!zip -j plantcare_pi5_models.zip exported_models/*
from google.colab import files
files.download("plantcare_pi5_models.zip")
print("Downloaded plantcare_pi5_models.zip! Transfer these files to ~/plantCare/models/ on your Raspberry Pi 5.")